<a href="https://colab.research.google.com/github/Rhuan-Messias/T-picosIA/blob/main/v5_ml_cat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics

In [ ]:
# Instale o TensorFlow se você não tiver (já vem pré-instalado no Colab)
# !pip install tensorflow
import tensorflow as tf
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

print(f"TensorFlow version: {tf.__version__}")


In [ ]:
# Carregar o dataset
# O split separa a base: 'train[:80%]' para treino e 'train[80%:]' para validação
(train_ds, validation_ds), info = tfds.load(
    'cats_vs_dogs',
    split=['train[:80%]', 'train[80%:]'],
    with_info=True,
    as_supervised=True
)

# Exibir informações sobre o dataset
print(f"Nome do dataset: {info.name}")
print(f"Total de exemplos: {info.splits['train'].num_examples}")
print("Dataset carregado com sucesso!")


In [ ]:
import matplotlib.pyplot as plt
import tensorflow_datasets as tfds

# 1. Carrega o dataset (apenas o necessário para visualização)
ds = tfds.load('cats_vs_dogs', split='train', as_supervised=True)

# 2. Pega 4 exemplos aleatórios
# O buffer do shuffle garante que a escolha seja realmente aleatória
random_examples = ds.shuffle(1000).take(4)

# 3. Cria a figura para a matriz 2x2
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes = axes.flatten() # Achata a matriz para facilitar o loop

# Mapeamento de labels (0 = Gato, 1 = Cachorro)
class_names = {0: 'Gato', 1: 'Cachorro'}

# 4. Loop para exibir as imagens
for i, (image, label) in enumerate(random_examples):
    axes[i].imshow(image.numpy().astype("uint8"))
    axes[i].set_title(f"Label: {class_names[label.numpy()]}")
    axes[i].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# Carrega o modelo ResNet50 pré-treinado no ImageNet
model = ResNet50(weights='imagenet')
model.summary()


In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input, decode_predictions
import numpy as np
import matplotlib.pyplot as plt

# 1. Carregar o Modelo ResNet50
print("Carregando modelo...")
model = ResNet50(weights='imagenet')

# 2. Carregar uma imagem do dataset cats_vs_dogs
print("Carregando dataset...")
# O cache() e prefetch ajudam na performance
ds = tfds.load('cats_vs_dogs', split='train', as_supervised=True)
image, label = next(iter(ds.shuffle(1000).take(1)))

# 3. Processar a imagem
# Redimensiona para o tamanho exigido pela ResNet50 (224x224)
img_resized = tf.image.resize(image, (224, 224))

# CORREÇÃO APLICADA: Converte para numpy e força uma cópia para evitar o erro read-only
img_array = img_resized.numpy().copy()
img_array = np.expand_dims(img_array, axis=0)
img_array = preprocess_input(img_array)

# 4. Classificar
predictions = model.predict(img_array, verbose=0)
decoded_predictions = decode_predictions(predictions, top=3)[0]

# 5. Exibir Resultado
plt.imshow(image.numpy().astype("uint8"))
plt.axis('off')
plt.title(f"Label Original no Dataset: {'Gato' if label == 0 else 'Cachorro'}")
plt.show()

print("Previsão da ResNet50:")
for i, (imagenet_id, label_name, score) in enumerate(decoded_predictions):
    print(f"{i + 1}: {label_name} ({score*100:.1f}%)")

# Lógica para verificar se é gato
cat_labels = ['tabby', 'tiger_cat', 'persian_cat', 'siamese_cat', 'egyptian_cat',
              'lynx', 'leopard', 'jaguar', 'cheetah', 'lion', 'tiger',
              'snow_leopard', 'cougar', 'cat', 'domestic_cat', 'kitten']

is_cat = any(label_name.lower() in cat_labels for _, label_name, _ in decoded_predictions)

if is_cat:
    print("\n✅ Resultado: A IA identificou como um gato!")
else:
    print("\n❌ Resultado: A IA não identificou como um gato.")


In [ ]:
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
from ultralytics import YOLO
import numpy as np

# 1. Carregar o modelo YOLO11n (o modelo será baixado automaticamente na 1ª vez)
model = YOLO('yolo11n.pt')

# 2. Carregar uma imagem do dataset cats_vs_dogs
print("Carregando imagem do dataset...")
ds = tfds.load('cats_vs_dogs', split='train', as_supervised=True)
image_tensor, label = next(iter(ds.shuffle(1000).take(1)))

# Converter para formato que o YOLO aceita (numpy uint8)
image_np = image_tensor.numpy().astype("uint8")

# 3. Rodar a detecção
# classes=[15] filtra apenas para a classe 'cat' do dataset COCO,
# se quiser detectar tudo (cachorros também), remova o parâmetro classes.
print("Detectando...")
results = model(image_np, classes=[15])

# 4. Exibir o resultado
# O plot() do ultralytics retorna um array da imagem com as caixas desenhadas
res_plotted = results[0].plot()

plt.figure(figsize=(8, 8))
plt.imshow(res_plotted)
plt.axis('off')
plt.title(f"Detecção YOLO11n | Label Dataset: {'Gato' if label == 0 else 'Cachorro'}")
plt.show()

# Verificar se detectou algo
if len(results[0].boxes) > 0:
    print(f"✅ Gato detectado com confiança: {results[0].boxes.conf[0].item()*100:.1f}%")
else:
    print("❌ Nenhum gato foi detectado na imagem (ou o modelo focou em outra coisa).")
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
from ultralytics import YOLO
import numpy as np

# 1. Carregar o modelo YOLO11n (o modelo será baixado automaticamente na 1ª vez)
model = YOLO('yolo11n.pt')

# 2. Carregar uma imagem do dataset cats_vs_dogs
print("Carregando imagem do dataset...")
ds = tfds.load('cats_vs_dogs', split='train', as_supervised=True)
image_tensor, label = next(iter(ds.shuffle(1000).take(1)))

# Converter para formato que o YOLO aceita (numpy uint8)
image_np = image_tensor.numpy().astype("uint8")

# 3. Rodar a detecção
# classes=[15] filtra apenas para a classe 'cat' do dataset COCO,
# se quiser detectar tudo (cachorros também), remova o parâmetro classes.
print("Detectando...")
results = model(image_np, classes=[15])

# 4. Exibir o resultado
# O plot() do ultralytics retorna um array da imagem com as caixas desenhadas
res_plotted = results[0].plot()

plt.figure(figsize=(8, 8))
plt.imshow(res_plotted)
plt.axis('off')
plt.title(f"Detecção YOLO11n | Label Dataset: {'Gato' if label == 0 else 'Cachorro'}")
plt.show()

# Verificar se detectou algo
if len(results[0].boxes) > 0:
    print(f"✅ Gato detectado com confiança: {results[0].boxes.conf[0].item()*100:.1f}%")
else:
    print("❌ Nenhum gato foi detectado na imagem (ou o modelo focou em outra coisa).")
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
from ultralytics import YOLO
import numpy as np

# 1. Carregar o modelo YOLO11n (o modelo será baixado automaticamente na 1ª vez)
model = YOLO('yolo11n.pt')

# 2. Carregar uma imagem do dataset cats_vs_dogs
print("Carregando imagem do dataset...")
ds = tfds.load('cats_vs_dogs', split='train', as_supervised=True)
image_tensor, label = next(iter(ds.shuffle(1000).take(1)))

# Converter para formato que o YOLO aceita (numpy uint8)
image_np = image_tensor.numpy().astype("uint8")

# 3. Rodar a detecção
# classes=[15] filtra apenas para a classe 'cat' do dataset COCO,
# se quiser detectar tudo (cachorros também), remova o parâmetro classes.
print("Detectando...")
results = model(image_np, classes=[15])

# 4. Exibir o resultado
# O plot() do ultralytics retorna um array da imagem com as caixas desenhadas
res_plotted = results[0].plot()

plt.figure(figsize=(8, 8))
plt.imshow(res_plotted)
plt.axis('off')
plt.title(f"Detecção YOLO11n | Label Dataset: {'Gato' if label == 0 else 'Cachorro'}")
plt.show()

# Verificar se detectou algo
if len(results[0].boxes) > 0:
    print(f"✅ Gato detectado com confiança: {results[0].boxes.conf[0].item()*100:.1f}%")
else:
    print("❌ Nenhum gato foi detectado na imagem (ou o modelo focou em outra coisa).")


In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input, decode_predictions
from ultralytics import YOLO
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# --- CONFIGURAÇÃO INICIAL E CARREGAMENTO ---
print("Carregando modelos (ResNet50 e YOLO11n)...")
# ResNet50 (para Classificação)
model_resnet = ResNet50(weights='imagenet')

# YOLO11n (para Detecção)
model_yolo = YOLO('yolo11n.pt')

# Lista de labels de gatos da ImageNet para ResNet50
CAT_LABELS_IMAGENET = ['tabby', 'tiger_cat', 'persian_cat', 'siamese_cat', 'egyptian_cat',
                        'lynx', 'leopard', 'jaguar', 'cheetah', 'lion', 'tiger',
                        'snow_leopard', 'cougar', 'cat', 'domestic_cat', 'kitten']

# --- SORTEAR E CARREGAR IMAGEM ---
print("Sorteando imagem do dataset 'cats_vs_dogs'...")
ds = tfds.load('cats_vs_dogs', split='train', as_supervised=True)
image_tensor, label = next(iter(ds.shuffle(1000).take(1)))
image_np = image_tensor.numpy().astype("uint8")

# Preparar figura para exibição lado a lado (1 linha, 2 colunas)
fig, axes = plt.subplots(1, 2, figsize=(15, 7))

# --- 1. PROCESSAMENTO RESNET50 (CLASSIFICAÇÃO) ---
print("Rodando ResNet50...")
# Preparar imagem: Resize (224x224), cópia mutável e preprocess_input
img_resnet = tf.image.resize(image_tensor, (224, 224))
img_resnet_array = img_resnet.numpy().copy()
img_resnet_array = np.expand_dims(img_resnet_array, axis=0)
img_resnet_array = preprocess_input(img_resnet_array)

# Predição
preds = model_resnet.predict(img_resnet_array, verbose=0)
decoded_preds = decode_predictions(preds, top=1)[0][0] # Pega o top 1
score_resnet = decoded_preds[2]
label_resnet = decoded_preds[1]

# Verificação se é gato na ResNet
is_cat_resnet = label_resnet.lower() in CAT_LABELS_IMAGENET

# Exibição ResNet
axes[0].imshow(image_np)
axes[0].axis('off')
title_resnet = f"ResNet50: {label_resnet} ({score_resnet*100:.1f}%)"
color_resnet = 'green' if is_cat_resnet else 'red'
axes[0].set_title(title_resnet, fontsize=12, color=color_resnet, fontweight='bold')


# --- 2. PROCESSAMENTO YOLO11N (DETECÇÃO) ---
print("Rodando YOLO11n...")
# Predição (classes=[15] para gatos, classes=[16] para cachorros. Se quiser detectar ambos, remova classes=[15])
results_yolo = model_yolo(image_np, verbose=0, classes=[15, 16])

# Exibição YOLO
# results[0].plot() desenha a bounding box e a label na imagem
img_yolo_annotated = results_yolo[0].plot()

axes[1].imshow(img_yolo_annotated)
axes[1].axis('off')

# Lógica de título para YOLO
num_objects = len(results_yolo[0].boxes)
conf_yolo = results_yolo[0].boxes.conf[0].item() if num_objects > 0 else 0
label_yolo = results_yolo[0].names[int(results_yolo[0].boxes.cls[0].item())] if num_objects > 0 else "Nenhum"

title_yolo = f"YOLO11n: Detectou {num_objects} {label_yolo}(s)"
color_yolo = 'green' if num_objects > 0 else 'black'
axes[1].set_title(title_yolo, fontsize=12, color=color_yolo, fontweight='bold')

# --- EXIBIR RESULTADO FINAL LADO A LADO ---
plt.suptitle(f"Label Real no Dataset: {'Gato' if label == 0 else 'Cachorro'}", fontsize=14, y=0.95, fontweight='bold')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# Resumo impresso
print("-" * 30)
print(f"Resumo do Gato (Label: {label}):")
print(f"ResNet50 classificou como: {label_resnet} ({score_resnet*100:.1f}%) -> {'✅ É um gato' if is_cat_resnet else '❌ Não é um gato'}")
print(f"YOLO11n detectou: {num_objects} objetos. Confiança: {conf_yolo*100:.1f}%")


Exercício 5 começa aqui


In [ ]:
!pip install ultralytics -q
import tensorflow as tf
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input, decode_predictions
import tensorflow_datasets as tfds
from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os
import random
import shutil

In [ ]:
# 1. ResNet50 (Classificador)
model_resnet = ResNet50(weights='imagenet')

# 2. YOLO11n (Detector - Versão Nano, mais rápida)
model_yolo = YOLO('yolo11n.pt')

print("✅ ResNet50 e YOLO11n carregados com sucesso!")

In [ ]:
def duelo_modelos(img_path):
    # --- RESNET50 (Keras - Aceita verbose=0) ---
    img_res = tf.keras.preprocessing.image.load_img(img_path, target_size=(224, 224))
    x = tf.keras.preprocessing.image.img_to_array(img_res)
    x = np.expand_dims(x, axis=0).copy()
    x = preprocess_input(x)

    preds = model_resnet.predict(x, verbose=0)
    decoded = decode_predictions(preds, top=1)[0][0]

    cat_keywords = ['cat', 'tabby', 'tiger_cat', 'persian', 'siamese', 'egyptian_cat']
    resnet_achou = any(word in decoded[1].lower() for word in cat_keywords)

    # --- YOLO11n (Ultralytics - EXIGE verbose=False) ---
    # Mudamos de verbose=0 para verbose=False para corrigir o TypeError
    results = model_yolo(img_path, verbose=False)[0]

    img_cv = cv2.imread(img_path)
    img_cv = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
    yolo_achou = False

    # Processar detecções da YOLO
    for box in results.boxes:
        label_nome = results.names[int(box.cls[0])]
        if label_nome == 'cat':
            yolo_achou = True
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cv2.rectangle(img_cv, (x1, y1), (x2, y2), (255, 0, 0), 4)

    # --- EXIBIÇÃO LADO A LADO ---
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))

    # Lado da ResNet
    ax[0].imshow(Image.open(img_path))
    ax[0].set_title(f"ResNet: {'GATO' if resnet_achou else 'NÃO GATO'}\n({decoded[1]})")
    ax[0].axis('off')

    # Lado da YOLO
    ax[1].imshow(img_cv)
    ax[1].set_title(f"YOLO11: {'DETECTADO' if yolo_achou else 'NADA'}")
    ax[1].axis('off')

    plt.tight_layout()
    plt.show()

print("✅ Função 'duelo_modelos' corrigida com verbose=False!")

In [ ]:
# 1. Limpar e criar pastas
if os.path.exists("dataset_comparativo"): shutil.rmtree("dataset_comparativo")
os.makedirs("dataset_comparativo")

# 2. Carregar 50 imagens (25 gatos e 25 cães para testar o 'Não Gato')
print("⏳ Gerando dataset para o Exercício 5...")
ds_gatos = tfds.load('cats_vs_dogs', split='train[:25]', as_supervised=True)
ds_caes = tfds.load('cats_vs_dogs', split='train[1000:1025]', as_supervised=True)

def salvar_fotos(dataset, prefixo):
    for i, (img_tensor, _) in enumerate(dataset):
        path = f"dataset_comparativo/{prefixo}_{i}.jpg"
        Image.fromarray(img_tensor.numpy()).save(path)

salvar_fotos(ds_gatos, "gato")
salvar_fotos(ds_caes, "nao_gato")

print(f"✅ Dataset pronto em 'dataset_comparativo/'. Total: {len(os.listdir('dataset_comparativo'))} imagens.")

In [ ]:
# 1. Definir a pasta onde estão as fotos
pasta = "dataset_comparativo"

# 2. Listar todos os arquivos da pasta
todos_os_arquivos = [f for f in os.listdir(pasta) if f.endswith('.jpg')]

if not todos_os_arquivos:
    print("⚠️ A pasta está vazia! Rode o Bloco 1 primeiro.")
else:
    # 3. Sortear uma imagem aleatória da lista
    imagem_sorteada = random.choice(todos_os_arquivos)
    caminho_completo = os.path.join(pasta, imagem_sorteada)

    print(f"🎲 Sorteio do Teste: {imagem_sorteada}")
    print(f"---" * 10)

    # 4. Rodar o duelo com a imagem sorteada
    duelo_modelos(caminho_completo)